In [2]:
import pandas as pd
df_final = pd.read_csv("C:/Users/admin/Downloads/datasets/final_prediction_dataset.csv")

In [3]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1000 non-null   str    
 1   actual_liquidity       1000 non-null   int64  
 2   loan_inflows           1000 non-null   int64  
 3   borrowing_outflows     1000 non-null   float64
 4   repo_rate              1000 non-null   float64
 5   cp_rate                1000 non-null   float64
 6   liquidity_index        1000 non-null   float64
 7   alm_gap                1000 non-null   int64  
 8   funding_risk_exposure  1000 non-null   int64  
 9   variance               1000 non-null   int64  
 10  predicted_liquidity    1000 non-null   float64
 11  scenario               1000 non-null   str    
 12  breach_flag            1000 non-null   int64  
 13  liquidity_risk_score   1000 non-null   float64
dtypes: float64(6), int64(6), str(2)
memory usage: 125.1 KB


In [4]:

selected_scenario = "Stress"

scenario_view = df_final[df_final["scenario"] == selected_scenario]

scenario_view[
    [
        "date",
        "scenario",
        "actual_liquidity",
        "breach_flag",
        "liquidity_index",
        "liquidity_risk_score"   
        
    ]
].sort_values("date")

,date,scenario,actual_liquidity,breach_flag,liquidity_index,liquidity_risk_score
500,2024-01-01,Stress,3219110,1,1.47,381912.7
501,2024-01-02,Stress,3768307,1,1.11,123441.4
502,2024-01-03,Stress,3229084,1,0.85,158957.2
503,2024-01-04,Stress,4511566,0,0.61,287139.2
504,2024-01-05,Stress,3356330,1,0.65,329684.9
...,...,...,...,...,...,...
995,2025-05-10,Stress,4080761,0,0.51,168524.9
996,2025-05-11,Stress,2414466,1,1.50,291314.2
997,2025-05-12,Stress,4114855,1,1.18,-287191.4
998,2025-05-13,Stress,1445101,1,1.33,322785.4


In [5]:
daily_prediction = df_final.groupby("date")[
    [
        "predicted_liquidity",
        "liquidity_risk_score"
    ]
].mean().reset_index()

daily_prediction

,date,predicted_liquidity,liquidity_risk_score
0,2024-01-01,-526148.10,381912.7
1,2024-01-02,-2267005.65,123441.4
2,2024-01-03,-658449.85,158957.2
3,2024-01-04,669100.40,287139.2
4,2024-01-05,-563837.65,329684.9
...,...,...,...
495,2025-05-10,656823.50,168524.9
496,2025-05-11,-2239383.30,291314.2
497,2025-05-12,-616088.15,-287191.4
498,2025-05-13,-426712.90,322785.4


In [6]:

def liquidity_color(val):
    
    if val < 0:
        return 'background-color: red; color: white'
    
    elif val < 100000:
        return 'background-color: yellow; color: black'
    
    else:
        return 'background-color: lightgreen; color: black'

In [7]:
import numpy as np
df_final["Breach_Zone"] = np.where(
    df_final["predicted_liquidity"] < 0,
    "YES",
    "NO"
)

df_final[["date", "scenario", "predicted_liquidity", "Breach_Zone"]]

,date,scenario,predicted_liquidity,Breach_Zone
0,2024-01-01,Normal,-411557.0,YES
1,2024-01-02,Normal,-2099023.0,YES
2,2024-01-03,Normal,-535140.0,YES
3,2024-01-04,Normal,713498.0,NO
4,2024-01-05,Normal,-428369.0,YES
...,...,...,...,...
995,2025-05-10,Stress,628516.0,NO
996,2025-05-11,Stress,-2386093.6,YES
997,2025-05-12,Stress,-735457.3,YES
998,2025-05-13,Stress,-562661.8,YES


In [8]:
high_risk = df_final[
    (df_final["breach_flag"] > 0.7) 
]

high_risk[
    [
        "date",
        "scenario",
        "breach_flag"
        
    ]
]

,date,scenario,breach_flag
0,2024-01-01,Normal,1
1,2024-01-02,Normal,1
2,2024-01-03,Normal,1
4,2024-01-05,Normal,1
6,2024-01-07,Normal,1
...,...,...,...
992,2025-05-07,Stress,1
993,2025-05-08,Stress,1
996,2025-05-11,Stress,1
997,2025-05-12,Stress,1


In [9]:
df_final["liquidity_Interpretation"] = np.where(
    df_final["predicted_liquidity"] < 0,
    "Future liquidity crisis | Immediate action needed",
    "Liquidity stable"
)

In [10]:
df_final["liquidity_Interpretation"]

0      Future liquidity crisis | Immediate action needed
1      Future liquidity crisis | Immediate action needed
2      Future liquidity crisis | Immediate action needed
3                                       Liquidity stable
4      Future liquidity crisis | Immediate action needed
                             ...                        
995                                     Liquidity stable
996    Future liquidity crisis | Immediate action needed
997    Future liquidity crisis | Immediate action needed
998    Future liquidity crisis | Immediate action needed
999                                     Liquidity stable
Name: liquidity_Interpretation, Length: 1000, dtype: str

In [11]:
df_final["Breach_Interpretation"] = np.where(
    df_final["breach_flag"] > 0.7,
    "High risk environment | Need contingency planning",
    "Normal risk"
)

In [12]:
df_final["Breach_Interpretation"] 

0      High risk environment | Need contingency planning
1      High risk environment | Need contingency planning
2      High risk environment | Need contingency planning
3                                            Normal risk
4      High risk environment | Need contingency planning
                             ...                        
995                                          Normal risk
996    High risk environment | Need contingency planning
997    High risk environment | Need contingency planning
998    High risk environment | Need contingency planning
999                                          Normal risk
Name: Breach_Interpretation, Length: 1000, dtype: str

In [13]:
df_final["Stress_Interpretation"] = np.where(
    df_final["liquidity_risk_score"] > 0.7,
    "System sensitive to shocks | Poor resilience",
    "System resilient"
)

In [14]:
df_final["Stress_Interpretation"]

0      System sensitive to shocks | Poor resilience
1      System sensitive to shocks | Poor resilience
2      System sensitive to shocks | Poor resilience
3      System sensitive to shocks | Poor resilience
4      System sensitive to shocks | Poor resilience
                           ...                     
995    System sensitive to shocks | Poor resilience
996    System sensitive to shocks | Poor resilience
997                                System resilient
998    System sensitive to shocks | Poor resilience
999    System sensitive to shocks | Poor resilience
Name: Stress_Interpretation, Length: 1000, dtype: str

In [15]:


df_final["Liquidity_Gap"] = (df_final["predicted_liquidity"] - df_final["liquidity_risk_score"]).abs()

df_final["Gap_Interpretation"] = np.where(
    df_final["Liquidity_Gap"] > 500000,
    "High uncertainty | Risk of extreme events",
    "Stable forecast range"
)

In [16]:
df_final.columns

Index(['date', 'actual_liquidity', 'loan_inflows', 'borrowing_outflows',
       'repo_rate', 'cp_rate', 'liquidity_index', 'alm_gap',
       'funding_risk_exposure', 'variance', 'predicted_liquidity', 'scenario',
       'breach_flag', 'liquidity_risk_score', 'Breach_Zone',
       'liquidity_Interpretation', 'Breach_Interpretation',
       'Stress_Interpretation', 'Liquidity_Gap', 'Gap_Interpretation'],
      dtype='str')